# 6 · ReactorLab: un sensor virtual y un diagnóstico de averías

En AeroLab una avería se veía en los datos: la potencia caía, la vibración subía. En una planta química
**con control automático** no es tan fácil: los lazos de control corrigen las perturbaciones y dejan las
medidas **en su consigna**. Si el transmisor de presión deriva, el lazo de presión «corrige» una presión que
no existe… y la medida sigue clavada en la consigna. **El control tapa la avería.**

Lo que sí cambia es **lo que hace el control** (las válvulas) y **el resto de la planta** (caudales,
composición, coste). En este cuaderno:

* **A. Sensor virtual de presión** (regresión): calcula la presión a partir de las **demás** medidas. Si la
  presión medida y la virtual no cuadran, uno de los dos miente.
* **B. Diagnóstico** (clasificación): mira los últimos 30 s y dice **qué** avería es la más probable.

Datos: el conjunto `reactorlab` que ha recogido el flujo de Node-RED (una fila por segundo, con la campaña de
consignas y los escenarios).

In [ ]:
from labia import datos, guardar
import numpy as np, pandas as pd, matplotlib.pyplot as plt

df = datos("reactorlab")

# Segundos desde el último cambio de escenario (justo después de un cambio la planta aún está reaccionando)
tramo = (df.escenario != df.escenario.shift()).cumsum()
df["seg_desde_cambio"] = (df.ts - df.groupby(tramo).ts.transform("min")).dt.total_seconds()
df.groupby("escenario").size().to_frame("filas")

## El control tapa la avería

Primero, toda la campaña: cada cambio de consigna sacude la planta (picos de presión, de válvula y de coste).
Las averías están sombreadas. ¿Se distinguen a simple vista?

In [ ]:
COLORES = {"deriva-presion": "tab:red", "purga-sucia": "tab:orange",
           "analizador-congelado": "tab:purple", "ruido-nivel": "tab:green"}

def sombrear(ax, d=df):
    for (_, g) in d.groupby(tramo.loc[d.index]):
        e = g.escenario.iloc[0]
        if e in COLORES:
            ax.axvspan(g.ts.iloc[0], g.ts.iloc[-1], color=COLORES[e], alpha=.15, label=e)

def leyenda(ax):
    h, l = ax.get_legend_handles_labels(); d = dict(zip(l, h))
    ax.legend(d.values(), d.keys(), loc="upper left", fontsize=8, ncol=3)

fig, ejes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
ejes[0].plot(df.ts, df.presion, lw=.8, label="presión medida")
ejes[0].plot(df.ts, df.sp_presion, "k--", lw=.8, label="consigna")
ejes[0].set_ylabel("kPa")
ejes[1].plot(df.ts, df.v_purga, lw=.8, color="tab:brown"); ejes[1].set_ylabel("válvula de purga (%)")
ejes[2].plot(df.ts, df.coste, lw=.8, color="tab:gray"); ejes[2].set_ylabel("coste relativo")
for ax in ejes:
    sombrear(ax); ax.grid(alpha=.3)
leyenda(ejes[0]);

A simple vista, no. Con la **deriva del sensor de presión** la medida vuelve a la consigna después de cada
cambio, igual que en funcionamiento normal: el lazo de presión «corrige» un error que no existe. Lo vemos
mejor con un modelo.

## A · Sensor virtual de presión

Entradas: las válvulas, los caudales, el nivel y la composición. **No** usamos la presión medida (es lo que
queremos vigilar) ni las consignas (el control ya las refleja en las válvulas).

Entrenamos con funcionamiento **normal** y, para evaluar, separamos el **último 25 %** en el tiempo: así se
comprueba el modelo con un rato de planta que no ha visto (más honesto que barajar filas de segundos
consecutivos, que se parecen muchísimo entre sí).

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # la red no termina de converger: ya se ve en su error

ENTRADAS = ["v_f1", "v_f2", "v_purga", "v_producto",
            "caudal_f1", "caudal_f2", "caudal_purga", "caudal_producto",
            "nivel", "a_purga", "b_purga", "c_purga"]
TRANSICION = 60
normal = df[(df.escenario == "normal") & (df.seg_desde_cambio >= TRANSICION)]
corte = int(len(normal) * 0.75)
ent, prueba = normal.iloc[:corte], normal.iloc[corte:]

candidatos = {
    "lineal":          make_pipeline(StandardScaler(), LinearRegression()),
    "red neuronal":    make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(32, 16), max_iter=3000, random_state=0)),
    "bosque aleatorio": RandomForestRegressor(n_estimators=200, random_state=0),
}
for nombre, m in candidatos.items():
    m.fit(ent[ENTRADAS], ent.presion)
    print(f"{nombre:17s} MAE en prueba = {mean_absolute_error(prueba.presion, m.predict(prueba[ENTRADAS])):6.1f} kPa")

> **¿Por qué gana el modelo más sencillo?** La presión de un gas está ligada casi linealmente a los caudales que
> entran y salen. El bosque aleatorio **no sabe extrapolar**: fuera de lo que vio al entrenar devuelve el
> valor más parecido que conoce. La red neuronal necesita muchos más datos. Y con una avería la planta se va
> **fuera** de lo visto: justo donde un sensor virtual tiene que seguir funcionando.

In [ ]:
virtual = candidatos["lineal"]
df["presion_virtual"] = virtual.predict(df[ENTRADAS])
df["residuo"] = df.presion - df.presion_virtual

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df.ts, df.residuo, lw=.8)
sombrear(ax); ax.grid(alpha=.3); ax.set_ylabel("medida − virtual (kPa)"); leyenda(ax)
ax.set_title("Residuo del sensor virtual");

Ahora sí: en la deriva, la presión **medida** sigue a la consigna, pero la que calculan las demás medidas (la
**virtual**, la más parecida a la real) se va alejando hacia abajo.

In [ ]:
ini = df.ts[df.escenario == "deriva-presion"].min() - pd.Timedelta("90s")
fin = df.ts[df.escenario == "deriva-presion"].max() + pd.Timedelta("30s")
z = df[(df.ts >= ini) & (df.ts <= fin)]
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(z.ts, z.presion, label="presión medida")
ax.plot(z.ts, z.presion_virtual, label="presión virtual (modelo)")
ax.plot(z.ts, z.sp_presion, "k--", lw=.8, label="consigna")
sombrear(ax, z); ax.grid(alpha=.3); ax.set_ylabel("kPa"); leyenda(ax)
ax.set_title("Deriva del transmisor de presión: medida frente a virtual");

In [ ]:
ev = df[df.seg_desde_cambio >= TRANSICION]
ev.groupby("escenario").residuo.agg(mediana="median", medio="mean", maximo="max").round(1)

Publícalo. El nombre `reactorlab-presion` es el que usa el nodo **ia predecir** del flujo; las *entradas* son
claves de la fila plana de Node-RED.

In [ ]:
mae = float(mean_absolute_error(prueba.presion, virtual.predict(prueba[ENTRADAS])))
guardar(virtual, "reactorlab-presion", entradas=ENTRADAS, tipo="regresion", salida="presion",
        descripcion="Sensor virtual: presión calculada con válvulas, caudales, nivel y composición",
        metricas={"mae_kpa": round(mae, 1)})

## B · Diagnóstico de averías

Una sola fila dice poco (en lazo cerrado casi todo vuelve a la consigna). Un técnico miraría **un rato** de
señal: ¿el residuo crece?, ¿el analizador está quieto?, ¿el nivel salta? Calculamos 6 **indicadores** sobre una
ventana de 30 s (30 filas):

| Indicador | Qué mide |
|---|---|
| `res_mediana` | desvío **que se mantiene** entre la presión medida y la virtual |
| `res_pendiente` | si ese desvío crece o decrece (kPa/s) |
| `res_pico` | el pico más alto por encima de la mediana: transitorios que no cuadran |
| `a_variacion` | cuánto se ha movido el analizador (suma de saltos) |
| `nivel_salto` | salto medio del nivel de un segundo a otro |
| `vprod_salto` | salto medio de la válvula de producto (el lazo de nivel persiguiendo ruido) |

**Importante:** Node-RED calcula estos indicadores en tiempo real en la *function* «Residuo y ventana de 30 s».
Esta función hace **exactamente lo mismo**; si cambias una, cambia la otra.

> **¿Por qué mediana y no media?** Con la válvula de purga sucia el residuo da **picos** enormes en cada
> transitorio (mira la gráfica del residuo). Una media se dispara con un solo pico y se parecería a una deriva;
> la mediana no. Separar «desvío que se mantiene» (mediana) de «picos» (`res_pico`) es lo que permite
> distinguir las dos averías.

In [ ]:
N = 30
INDICADORES = ["res_mediana", "res_pendiente", "res_pico", "a_variacion", "nivel_salto", "vprod_salto"]

def ventanas(d, n=N):
    """Indicadores de los últimos n segundos para cada fila (igual que la function de Node-RED)."""
    d = d.copy()
    seg = (d.ts.diff().dt.total_seconds() > 5).cumsum()           # un hueco de más de 5 s corta la ventana
    w = np.arange(n) - (n - 1) / 2
    partes = []
    for _, g in d.groupby(seg):
        r = g.residuo.rolling(n)
        g["res_mediana"] = r.median()
        g["res_pendiente"] = r.apply(lambda a: np.dot(w, a) / np.dot(w, w), raw=True)
        g["res_pico"] = r.max() - g.res_mediana
        g["a_variacion"] = g.a_purga.diff().abs().rolling(n - 1).sum()
        g["nivel_salto"] = g.nivel.diff().abs().rolling(n - 1).mean()
        g["vprod_salto"] = g.v_producto.diff().abs().rolling(n - 1).mean()
        partes.append(g)
    return pd.concat(partes).dropna(subset=INDICADORES)

dv = ventanas(df)
# solo ventanas «puras»: los 30 s dentro del mismo escenario y ya pasada la transición
dv = dv[dv.seg_desde_cambio >= TRANSICION]
dv.groupby("escenario")[INDICADORES].median().round(2)

Para evaluar, en **cada** escenario entrenamos con el primer 70 % de su tiempo y probamos con el 30 % final.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score

dv["orden"] = dv.groupby(tramo.loc[dv.index]).cumcount() / dv.groupby(tramo.loc[dv.index]).ts.transform("size")
ent_c, prueba_c = dv[dv.orden < 0.7], dv[dv.orden >= 0.7]

clf = RandomForestClassifier(n_estimators=200, min_samples_leaf=3, random_state=0)
clf.fit(ent_c[INDICADORES], ent_c.escenario)
pred = clf.predict(prueba_c[INDICADORES])
acierto = accuracy_score(prueba_c.escenario, pred)
print(f"acierto en prueba: {100 * acierto:.1f} %")
ConfusionMatrixDisplay.from_predictions(prueba_c.escenario, pred, xticks_rotation=30)
plt.xlabel("según el modelo"); plt.ylabel("realidad");

In [ ]:
pd.Series(clf.feature_importances_, index=INDICADORES).sort_values().plot.barh(figsize=(7, 3))
plt.title("¿En qué se fija el clasificador?");

Antes de publicar, lo reentrenamos con **todas** las ventanas (cuantos más ejemplos, mejor).

In [ ]:
clf.fit(dv[INDICADORES], dv.escenario)
guardar(clf, "reactorlab-averias", entradas=INDICADORES, tipo="clasificacion", salida="escenario",
        descripcion="Diagnóstico de averías con 6 indicadores de una ventana de 30 s",
        metricas={"acierto_pct": round(100 * acierto, 1), "ventana_s": N})

## ✏️ Tu turno

1. Cambia `res_mediana` por la **media** de la ventana (`r.mean()`) y `res_pico` por la desviación típica
   (`r.std(ddof=0)`). ¿Qué dos averías se confunden ahora? ¿Por qué?
2. Quita `res_mediana`, `res_pendiente` y `res_pico` (deja solo los indicadores que no usan el sensor virtual).
   ¿Qué avería deja de verse? ¿Qué aporta el sensor virtual?
3. Cambia `N = 10` y `N = 60`. ¿Mejora el acierto? ¿Qué pierdes en tiempo de detección? (Si lo publicas, tienes
   que cambiar también `N` en la function de Node-RED.)
4. Detén la **campaña de consignas** y deja la planta quieta 5 minutos. ¿Qué dice el diagnóstico? (Pista:
   `a_variacion`.) ¿Qué condición tiene que cumplirse en la planta para que este clasificador sirva?
5. El clasificador solo conoce las 4 averías que le enseñaste. ¿Qué dirá ante una válvula atascada? ¿Cómo lo
   resolverías (pista: cuaderno 02, detector de anomalías)?